In [1]:
import re
import numpy as np
import pandas as pd
import copy
import ast

In [2]:
df = pd.read_excel('atom_list.xlsx',skiprows=lambda x: x not in [0, 10, 6, 7] )
print(df)

                         atom                atom example   atom pattern  \
0           as(D, (V, V'), X)           as(0,(mbf,mgt),2)         as\(.*   
1  dl(arrival_v(D, V, X), T )  dl(arrival_v(1,lbm,2),165)  dl\(arrival.*   
2     dl(start_v(D, V, X), T)    dl(start_v(0,mbf,4),166)  dl\(start_v.*   

   number of terms term_not_convert sort_order  sort reserve order  
0                5        [0, 2, 3]     [4, 1]                 NaN  
1                5           [0, 2]     [3, 1]                 NaN  
2                5           [0, 2]      [3,1]                 NaN  


In [3]:
data_raw = 'as(0,(mbf,crj),0) as(1,(lbm,mbf),0) as(2,(lbm,mbf),0) as(3,(tci,mgt),0) as(4,(mbf,crj),0) as(5,(mbf,crj),0) as(6,(mbf,crj),0) as(7,(mbf,crj),0) as(8,(tci,mgt),0) as(9,(lbm,mbf),0) as(10,(crj,mbf),0) as(11,(crj,mbf),0) dl(start_v(0,crj,1),93) dl(ready_v(4,crj,0),94) dl(ready_v(6,crj,0),94) dl(ready_v(7,crj,0),94) dl(ready_v(0,crj,1),68) dl(ready_v(1,crj,1),94) dl(ready_v(4,crj,1),68) dl(ready_v(5,crj,1),68) dl(ready_v(6,crj,1),68) dl(ready_v(7,crj,1),68) dl(ready_v(8,crj,1),284) dl(ready_v(1,crj,2),188) dl(ready_v(6,crj,2),284) dl(ready_v(7,crj,2),94) dl(ready_v(8,crj,2),228) dl(ready_v(1,crj,3),188) dl(ready_v(5,crj,3),284) dl(ready_v(6,crj,3),360) dl(ready_v(7,crj,3),228) dl(ready_v(1,crj,4),188) dl(ready_v(3,crj,4),228) dl(ready_v(5,crj,4),334) dl(ready_v(11,crj,5),284) dl(ready_v(11,crj,6),334) dl(ready_v(3,lbm,1),91) dl(ready_v(6,lbm,1),91) dl(ready_v(3,lbm,2),91) dl(ready_v(4,lbm,2),91) dl(ready_v(6,lbm,2),214) dl(ready_v(9,lbm,2),91) dl(ready_v(10,lbm,2),188) dl(ready_v(11,lbm,2),188) dl(ready_v(0,lbm,3),241) dl(ready_v(1,lbm,3),91) dl(ready_v(2,lbm,3),91) dl(ready_v(3,lbm,3),91) dl(ready_v(4,lbm,3),321) dl(ready_v(5,lbm,3),91) dl(ready_v(6,lbm,3),91) dl(ready_v(7,lbm,3),91) dl(ready_v(8,lbm,3),91) dl(ready_v(10,lbm,3),188) dl(ready_v(11,lbm,3),188) dl(ready_v(1,lbm,4),91) dl(ready_v(3,lbm,4),91) dl(ready_v(7,lbm,4),374) dl(ready_v(11,lbm,4),188) dl(ready_v(1,lbm,5),334) dl(ready_v(6,lbm,5),91) dl(ready_v(8,lbm,5),91) dl(ready_v(10,lbm,5),91) dl(ready_v(11,lbm,5),188) dl(ready_v(1,lbm,6),91) dl(ready_v(8,lbm,6),91) dl(ready_v(10,lbm,8),91) dl(ready_v(2,mgt,1),194) dl(ready_v(3,mgt,1),148) dl(ready_v(4,mgt,1),194) dl(ready_v(8,mgt,1),148) dl(ready_v(2,mgt,2),268) dl(ready_v(3,mgt,2),148) dl(ready_v(4,mgt,2),148) dl(ready_v(7,mgt,2),148) dl(ready_v(3,mgt,3),148) dl(ready_v(5,mgt,3),254) dl(ready_v(10,mgt,3),194) dl(ready_v(10,mgt,4),361) dl(ready_v(9,mgt,7),194) dl(ready_v(9,mgt,8),268) dl(start_v(0,mbf,2),196) dl(ready_v(2,mbf,0),41) dl(ready_v(10,mbf,0),41) dl(ready_v(11,mbf,0),41) dl(ready_v(0,mbf,1),41) dl(ready_v(1,mbf,1),135) dl(ready_v(2,mbf,1),135) dl(ready_v(5,mbf,1),41) dl(ready_v(6,mbf,1),41) dl(ready_v(9,mbf,1),135) dl(ready_v(10,mbf,1),68) dl(ready_v(11,mbf,1),68) dl(ready_v(0,mbf,2),121) dl(ready_v(2,mbf,2),41) dl(ready_v(5,mbf,2),121) dl(ready_v(9,mbf,2),135) dl(ready_v(11,mbf,2),221) dl(ready_v(0,mbf,3),221) dl(ready_v(2,mbf,3),41) dl(ready_v(9,mbf,3),135) dl(ready_v(11,mbf,3),221) dl(ready_v(0,mbf,4),361) dl(ready_v(3,mbf,4),221) dl(ready_v(9,mbf,4),135) dl(ready_v(1,mbf,5),41) dl(ready_v(3,mbf,5),281) dl(ready_v(9,mbf,5),135) dl(ready_v(0,mbf,6),41) dl(ready_v(9,mbf,6),135) dl(ready_v(10,mbf,6),41) dl(ready_v(11,mbf,6),41) dl(ready_v(1,mbf,7),41) dl(ready_v(9,mbf,7),135) dl(start_v(0,lbm,3),316) dl(start_v(0,mbf,0),40) dl(start_v(1,mbf,1),160) dl(start_v(1,crj,2),173) dl(start_v(1,crj,3),173) dl(start_v(1,crj,4),283) dl(start_v(1,lbm,0),90) dl(start_v(2,mbf,1),220) dl(start_v(2,lbm,0),90) dl(start_v(3,mgt,1),133) dl(start_v(3,mgt,2),133) dl(start_v(3,mgt,3),193) dl(start_v(3,crj,4),253) dl(start_v(3,tci,0),100) dl(start_v(4,crj,1),113) dl(start_v(4,mgt,2),263) dl(start_v(4,mbf,0),40) dl(start_v(5,crj,1),93) dl(start_v(5,mbf,2),206) dl(start_v(5,mgt,3),299) dl(start_v(5,mbf,0),40) dl(start_v(6,crj,1),163) dl(start_v(6,lbm,2),309) dl(start_v(6,mbf,0),40) dl(start_v(7,crj,1),113) dl(start_v(7,mgt,2),193) dl(start_v(7,crj,3),323) dl(start_v(7,mbf,0),40) dl(start_v(8,mgt,1),193) dl(start_v(8,tci,0),100) dl(start_v(9,mbf,1),120) dl(start_v(9,mbf,2),120) dl(start_v(9,mbf,3),120) dl(start_v(9,mbf,4),120) dl(start_v(9,mbf,5),120) dl(start_v(9,mbf,6),120) dl(start_v(9,mbf,7),220) dl(start_v(9,lbm,0),90) dl(start_v(10,mbf,1),143) dl(start_v(10,lbm,2),173) dl(start_v(10,lbm,3),303) dl(start_v(10,crj,0),40) dl(start_v(11,mbf,1),143) dl(start_v(11,lbm,2),173) dl(start_v(11,lbm,3),173) dl(start_v(11,lbm,4),173) dl(start_v(11,lbm,5),283) dl(start_v(11,crj,0),40) dl(arrival_v(0,crj,0),53) dl(arrival_v(1,mbf,0),120) dl(arrival_v(2,mbf,0),120) dl(arrival_v(3,mgt,0),133) dl(arrival_v(4,crj,0),53) dl(arrival_v(5,crj,0),53) dl(arrival_v(6,crj,0),53) dl(arrival_v(7,crj,0),53) dl(arrival_v(8,mgt,0),133) dl(arrival_v(9,mbf,0),120) dl(arrival_v(10,mbf,0),53) dl(arrival_v(11,mbf,0),53) dl(arrival_v(3,mgt,1),133) dl(arrival_v(9,mbf,1),120) dl(arrival_v(0,mbf,1),106) dl(arrival_v(1,crj,1),173) dl(arrival_v(2,mgt,1),253) dl(arrival_v(4,mgt,1),133) dl(arrival_v(5,mbf,1),106) dl(arrival_v(6,lbm,1),199) dl(arrival_v(7,mgt,1),133) dl(arrival_v(8,crj,1),213) dl(arrival_v(10,lbm,1),173) dl(arrival_v(11,lbm,1),173) dl(arrival_v(3,mgt,2),133) dl(arrival_v(6,crj,2),345) dl(arrival_v(7,crj,2),213) dl(arrival_v(9,mbf,2),120) dl(arrival_v(0,lbm,2),226) dl(arrival_v(1,crj,2),173) dl(arrival_v(4,lbm,2),306) dl(arrival_v(5,mgt,2),239) dl(arrival_v(10,lbm,2),173) dl(arrival_v(11,lbm,2),173) dl(arrival_v(0,mbf,3),346) dl(arrival_v(1,crj,3),173) dl(arrival_v(3,crj,3),213) dl(arrival_v(5,crj,3),319) dl(arrival_v(7,lbm,3),359) dl(arrival_v(9,mbf,3),120) dl(arrival_v(10,mgt,3),346) dl(arrival_v(11,lbm,3),173) dl(arrival_v(1,lbm,4),319) dl(arrival_v(3,mbf,4),266) dl(arrival_v(9,mbf,4),120) dl(arrival_v(11,lbm,4),173) dl(arrival_v(9,mbf,5),120) dl(arrival_v(11,crj,5),319) dl(arrival_v(9,mbf,6),120) dl(arrival_v(9,mgt,7),253) as(11,(mbf,lbm),1) as(10,(mbf,lbm),1) as(8,(mgt,crj),1) as(7,(crj,mgt),1) as(6,(crj,lbm),1) as(5,(crj,mbf),1) as(4,(crj,mgt),1) as(2,(mbf,mgt),1) as(1,(mbf,crj),1) as(0,(crj,mbf),1) as(9,(mbf,mbf),1) as(3,(mgt,mgt),1) as(7,(mgt,crj),2) as(6,(lbm,crj),2) as(5,(mbf,mgt),2) as(4,(mgt,lbm),2) as(0,(mbf,lbm),2) as(11,(lbm,lbm),2) as(10,(lbm,lbm),2) as(1,(crj,crj),2) as(9,(mbf,mbf),2) as(3,(mgt,mgt),2) as(10,(lbm,mgt),3) as(7,(crj,lbm),3) as(5,(mgt,crj),3) as(3,(mgt,crj),3) as(0,(lbm,mbf),3) as(11,(lbm,lbm),3) as(9,(mbf,mbf),3) as(1,(crj,crj),3) as(3,(crj,mbf),4) as(1,(crj,lbm),4) as(11,(lbm,lbm),4) as(9,(mbf,mbf),4) as(11,(lbm,crj),5) as(9,(mbf,mbf),5) as(9,(mbf,mbf),6) as(9,(mbf,mgt),7) as_w(3,(mgt,mgt),1,0) as_w(9,(mbf,mbf),1,0) as_w(3,(mgt,mgt),2,0) as_w(9,(mbf,mbf),2,0) as_w(1,(crj,crj),2,0) as_w(10,(lbm,lbm),2,0) as_w(11,(lbm,lbm),2,0) as_w(1,(crj,crj),3,0) as_w(9,(mbf,mbf),3,0) as_w(11,(lbm,lbm),3,0) as_w(9,(mbf,mbf),4,0) as_w(11,(lbm,lbm),4,0) as_w(9,(mbf,mbf),5,0) as_w(9,(mbf,mbf),6,0) as_w(0,(crj,mbf),1,0) as_w(1,(crj,lbm),4,0) as_w(2,(lbm,mbf),0,0) as_w(2,(mbf,mgt),1,0) as_w(3,(mgt,crj),3,3) as_w(4,(mbf,crj),0,0) as_w(5,(mbf,crj),0,0) as_w(5,(crj,mbf),1,0) as_w(6,(mbf,crj),0,0) as_w(7,(mbf,crj),0,0) as_w(9,(lbm,mbf),0,0) as_w(11,(crj,mbf),0,2) as_w(4,(mgt,lbm),2,10) as_w(10,(mbf,lbm),1,6) as_w(11,(mbf,lbm),1,5) as_w(0,(mbf,lbm),2,6) as_w(6,(crj,lbm),1,9) as_w(7,(crj,lbm),3,9) as_w(10,(lbm,mgt),3,10) as_w(3,(tci,mgt),0,6) as_w(8,(tci,mgt),0,10) as_w(5,(mbf,mgt),2,10) as_w(9,(mbf,mgt),7,7) as_w(4,(crj,mgt),1,10) as_w(7,(crj,mgt),1,7) as_w(1,(lbm,mbf),0,9) as_w(0,(lbm,mbf),3,10) as_w(10,(crj,mbf),0,8) as_w(3,(crj,mbf),4,10) as_w(6,(lbm,crj),2,10) as_w(11,(lbm,crj),5,7) as_w(8,(mgt,crj),1,4) as_w(7,(mgt,crj),2,4) as_w(5,(mgt,crj),3,9) as_w(0,(mbf,crj),0,7) as_w(1,(mbf,crj),1,6) passengers_served(10,(mgt,lbm)) passengers_served(0,(tci,lbm)) passengers_served(17,(mbf,lbm)) passengers_served(18,(crj,lbm)) passengers_served(10,(lbm,mgt)) passengers_served(16,(tci,mgt)) passengers_served(17,(mbf,mgt)) passengers_served(17,(crj,mgt)) passengers_served(0,(lbm,tci)) passengers_served(0,(mgt,tci)) passengers_served(0,(mbf,tci)) passengers_served(0,(crj,tci)) passengers_served(19,(lbm,mbf)) passengers_served(0,(mgt,mbf)) passengers_served(0,(tci,mbf)) passengers_served(20,(crj,mbf)) passengers_served(17,(lbm,crj)) passengers_served(20,(mgt,crj)) passengers_served(0,(tci,crj)) passengers_served(13,(mbf,crj))'


choose_atom_row = [i for i in range(0, len(df))]
atom_pattern=[df.loc[i, 'atom pattern'] for i in choose_atom_row]
nb_term = [df.loc[i, 'number of terms'] for i in choose_atom_row]
term_int_not_change = [ast.literal_eval(df.loc[i, 'term_not_convert']) for i in choose_atom_row]
# reverse= [l_reverse[i] for i in choose_atom_row]
sort_order= [ast.literal_eval(df.loc[i, 'sort_order']) for i in choose_atom_row]



In [29]:

out = []
i=0
for p in range(0, len(atom_pattern)):
    data = re.sub(' ', '\n', data_raw)
    data = re.findall(atom_pattern[p], data)

    data = [re.sub("dl\(", "", i) for i in data]
    data = [re.sub("\(", ",", i) for i in data]
    data = [re.sub("\)", "", i) for i in data]
    data = [re.sub(",,", ",", i) for i in data]

    data = [i.split(",") for i in data]
    # print(data)
    data = [[int(i) if k not in term_int_not_change[p] else i for i, k in zip(j,range(0, nb_term[p]))] for j in data]
    # print(data)
    for s, r in zip(sort_order[p], range(0, len(sort_order[p]))):
        # reverse=reverse[p][r]
        # print(sort_order[p])
        # print(len(sort_order[p]))
        # print(data)
        # print(s)
        # print(data[0][s])
        data = sorted(data, key = lambda x:x[s])
        
    if p == 0: 
        AS = pd.DataFrame(data)
    if p == 1:
        ARR = pd.DataFrame(data)
    if p == 2:
        STA = pd.DataFrame(data)
# Name of the output text file
# output_file = 'trajectory.txt'

# # Open the text file in write mode
# with open(output_file, 'w') as file:
#     # Iterate over each row in the list
#     for row in out:
#         # Convert each element to string and join them with tab separator
#         line = '\t'.join(map(str, row))
#         # Write the line to the text file
#         file.write(line + '\n')

# print("Text file created successfully:", output_file)
traj = pd.DataFrame()
for i in range(0, len(AS)):
#    traj = pd.concat([traj, AS.loc[i:i]], sort=False) 
#    traj = pd.concat([traj, STA.loc[i:i]], sort=False)
#    traj = pd.concat([traj, ARR.loc[i:i]], sort=False)
    traj = pd.concat([traj, pd.DataFrame([[STA.iloc[i, 1], STA.iloc[i, 2], 
                                           ARR.iloc[i, 2], STA.iloc[i, 4]]], columns=['AgentID', 'Origin', 'Destination', 'DepartureTime'])], sort=False)
# traj = traj.drop(traj.columns[3], axis=1)
# traj = traj.drop_duplicates(subset=[1, 4])
traj = traj[traj['Origin'] != traj['Destination']]
flightID = [i for i in range(0, len(traj))]
traj.insert(0, 'FlightID', flightID)

traj

traj.to_csv(r'output.csv', index=None)

In [6]:
print(df.index.name)

None


In [14]:
STA.iloc[2,2]

'mbf'

In [19]:
pd.DataFrame([1,2,3])

,0
0,1
1,2
2,3
